## 1. 필요 라이브러리 호출

In [68]:
# 환경설정
import os
import sys
import time
from tqdm import tqdm
import nest_asyncio
nest_asyncio.apply()
from dotenv import load_dotenv

load_dotenv()
# duckdb
import duckdb

# 데이터 전처리
import pandas as pd
import numpy as np
import polars as pl
from datetime import datetime, timedelta
from copy import deepcopy
# 데이터 수집
import requests
from bs4 import BeautifulSoup

# VectorDB 저장
from hashlib import md5
from langchain_community.vectorstores.utils import filter_complex_metadata # ChromaDB가 제공하지 못하는 데이터 형태를 자동으로 string처리


## LLM 활용
from summary_function import NewsSummaryAgent
# LLM 활용을 위한 dict형태 구축
from collections import defaultdict

# langchain 계열
from langchain_core.documents import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

from langchain_openai import ChatOpenAI

# 1. LLM 모델 세팅 (OpenAI)
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
#from summary_function_openai import NewsSummaryAgent, build_summary_graph  # 너가 만든 것
from summart_function_openai_2 import NewsSummaryAgent, build_summary_graph  # 너가 만든 것
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.3)

# ㄱRe-ranker 모델 활용
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder


## 2. ETF 목록 가져오기

In [2]:
ETF_conn = duckdb.connect('../DB/ETF.db')

In [3]:
ETF_df = ETF_conn.execute('select * from IRP_ETF_COMPOSE_table').fetchdf()
ETF_conn.close()

In [4]:
ETF_df = ETF_df.map(lambda x : x.strip())

In [5]:
ETF_df_task_1 = ETF_df[ETF_df['구성종목 종목명'] != "설정현금액"]
ETF_df_task_1 = ETF_df_task_1[ETF_df_task_1['구성종목 종목명'] != "원화현금"]
ETF_df_task_1 = ETF_df_task_1[1:]

In [6]:
# 원하는 컬럼 필터링
ETF_df_task_2 = ETF_df_task_1[['ETF 종목명','구성종목 표준코드','구성종목 종목명','편입비율']]

In [7]:
# 구성종목 중 상위 5개 추출
ETF_df_task_3 = ETF_df_task_2.sort_values(by=['ETF 종목명','편입비율'], ascending=False)

In [8]:
# 종목별 상위 5개
ETF_df_task_4= ETF_df_task_3.groupby('ETF 종목명').head(5)

In [9]:
ETF_df_task_4

,ETF 종목명,구성종목 표준코드,구성종목 종목명,편입비율
7839,파워 코스피100,KR7005930003,삼성전자,22.979477
7899,파워 코스피100,KR7105560007,KB금융,2.846638
7856,파워 코스피100,KR7012450003,한화에어로스페이스,2.680652
7876,파워 코스피100,KR7035420009,NAVER,2.595262
7836,파워 코스피100,KR7005380001,현대차,2.280681
...,...,...,...,...
59727,1Q 25-08 회사채(A+이상)액티브,KR6079317D93,JB 우리캐피탈488-3(지),8.851804
59728,1Q 25-08 회사채(A+이상)액티브,KR6095923D95,현대커머셜485-3(지),8.850334
59723,1Q 25-08 회사채(A+이상)액티브,KR6023788D91,신한캐피탈487-2,8.844456
59722,1Q 25-08 회사채(A+이상)액티브,KR601945CD90,아이비케이캐피탈290-7,8.842961


In [10]:
ETF_df_task_4[ETF_df_task_4['ETF 종목명'].str.contains('반도체')]

,ETF 종목명,구성종목 표준코드,구성종목 종목명,편입비율
61034,WON 반도체밸류체인액티브,KR7402340004,SK스퀘어,4.673431
61026,WON 반도체밸류체인액티브,KR7039030002,이오테크닉스,4.617986
61027,WON 반도체밸류체인액티브,KR7058470006,리노공업,4.522346
61019,WON 반도체밸류체인액티브,KR7000150003,두산,4.518261
61030,WON 반도체밸류체인액티브,KR7140860008,파크시스템스,4.513108
...,...,...,...,...
59981,ACE AI반도체포커스,KR7000660001,SK하이닉스,25.755918
59985,ACE AI반도체포커스,KR7005930003,삼성전자,24.736614
59992,ACE AI반도체포커스,KR7042700005,한미반도체,24.473018
59998,ACE AI반도체포커스,KR7140860008,파크시스템스,1.519344


In [11]:
ETF_df_task_4[ETF_df_task_4['ETF 종목명'].str.contains('전지')]

,ETF 종목명,구성종목 표준코드,구성종목 종목명,편입비율
44379,TIGER 글로벌리튬&2차전지SOLACTIVE(합성),KRYZTRSEAH18,글로벌리튬2차전지 TRS 241017-18,8.990678
44380,TIGER 글로벌리튬&2차전지SOLACTIVE(합성),KRYZTRSF1E02,글로벌리튬2차전지 TRS 250114-02,32.05569
44381,TIGER 글로벌리튬&2차전지SOLACTIVE(합성),KRYZTRSF5E01,글로벌리튬2차전지 TRS 250514-01,27.325525
44378,TIGER 글로벌리튬&2차전지SOLACTIVE(합성),KRYZTRSE7Q01,글로벌리튬2차전지 TRS 240724-01,2.877921
44377,TIGER 글로벌리튬&2차전지SOLACTIVE(합성),KRYZTRSE7C04,글로벌리튬2차전지 TRS 240712-04,14.624746
...,...,...,...,...
43926,ACE 2차전지&친환경차액티브,KR7012330007,현대모비스,8.46777
43920,ACE 2차전지&친환경차액티브,KR7005380001,현대차,8.23665
43914,ACE 2차전지&친환경차액티브,KR7000270009,기아,8.090785
43921,ACE 2차전지&친환경차액티브,KR7005490008,POSCO홀딩스,7.842668


In [12]:
ETF_df_task_4[ETF_df_task_4['ETF 종목명'].str.contains('자율주행')]

,ETF 종목명,구성종목 표준코드,구성종목 종목명,편입비율
44360,TIGER 글로벌자율주행&전기차SOLACTIVE,US5949181045,MICROSOFT CORP,3.65966
44363,TIGER 글로벌자율주행&전기차SOLACTIVE,US67066G1040,NVIDIA CORP,3.405895
44366,TIGER 글로벌자율주행&전기차SOLACTIVE,US7475251036,QUALCOMM INC,2.930486
44342,TIGER 글로벌자율주행&전기차SOLACTIVE,US02079K3059,ALPHABET INC-CL A,2.869684
44318,TIGER 글로벌자율주행&전기차SOLACTIVE,JP3633400001,TOYOTA MOTOR CORP,2.808817
43543,KODEX 자율주행액티브,KR7012330007,현대모비스,8.590932
43570,KODEX 자율주행액티브,KR7307950006,현대오토에버,7.626326
43529,KODEX 자율주행액티브,KR7000660001,SK하이닉스,5.967903
43556,KODEX 자율주행액티브,KR7086280005,현대글로비스,4.925796
43531,KODEX 자율주행액티브,KR7005380001,현대차,4.391058


In [13]:
ETF_df_task_4[ETF_df_task_4['ETF 종목명'].str.contains('금융')]

,ETF 종목명,구성종목 표준코드,구성종목 종목명,편입비율
34483,TIGER 25-12 금융채(AA-이상),KR6205498EC7,하나카드273,4.988738
34398,TIGER 25-12 금융채(AA-이상),KR6005273F23,아이엠뱅크46-02이12A-21,4.10297
34473,TIGER 25-12 금융채(AA-이상),KR6140178EB5,케이비국민카드421-1,3.300191
34461,TIGER 25-12 금융채(AA-이상),KR6079314EA3,JB 우리캐피탈524-1(지),2.485386
34475,TIGER 25-12 금융채(AA-이상),KR6145763DC9,BNK캐피탈338-3,2.48222
7606,TIGER 200 금융,KR7316140003,우리금융지주,7.825771
7587,TIGER 200 금융,KR7000810002,삼성화재,7.048532
7595,TIGER 200 금융,KR7032830002,삼성생명,5.727802
7607,TIGER 200 금융,KR7323410001,카카오뱅크,5.180046
7602,TIGER 200 금융,KR7138040001,메리츠금융지주,4.741424


## 3. 고객 데이터 시나리오
 - 고객 데이터 생성

In [14]:
Customer_A = ETF_df_task_4[ETF_df_task_4['ETF 종목명'].isin(['ACE AI반도체포커스','ACE 2차전지&친환경차액티브','KODEX 자율주행액티브','RISE 200금융'])]

In [15]:
Customer_A = Customer_A.reset_index().drop('index',axis = 1)

In [16]:
Customer_A

,ETF 종목명,구성종목 표준코드,구성종목 종목명,편입비율
0,RISE 200금융,KR7316140003,우리금융지주,7.848761
1,RISE 200금융,KR7000810002,삼성화재,7.198696
2,RISE 200금융,KR7032830002,삼성생명,5.759096
3,RISE 200금융,KR7323410001,카카오뱅크,5.192032
4,RISE 200금융,KR7138040001,메리츠금융지주,4.756095
5,KODEX 자율주행액티브,KR7012330007,현대모비스,8.590932
6,KODEX 자율주행액티브,KR7307950006,현대오토에버,7.626326
7,KODEX 자율주행액티브,KR7000660001,SK하이닉스,5.967903
8,KODEX 자율주행액티브,KR7086280005,현대글로비스,4.925796
9,KODEX 자율주행액티브,KR7005380001,현대차,4.391058


## 4. 고객 데이터 저장

In [24]:
con = duckdb.connect('../DB/Customer.db')

# Pandas DataFrame을 DuckDB에서 참조할 수 있도록 등록
con.register('temp_df', Customer_A)

# 테이블이 없다면 생성
con.execute("""
    CREATE TABLE IF NOT EXISTS Customers AS
    SELECT * FROM temp_df LIMIT 0
""")

# 데이터 삽입
con.execute("INSERT INTO Customers SELECT * FROM temp_df")

# 정리
con.unregister('temp_df')
con.close()

## 5. 고객이 보유하고 있는 데이터를 DB에 저장하기

In [77]:
import asyncio
import aiohttp
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin,urlparse
import duckdb

# ▶ 헤더
headers = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/137.0.0.0 Safari/537.36"
}

# ▶ 기사 리스트 파싱 함수
async def get_article_urls(session, page_url):
    try:
        async with session.get(page_url, headers=headers) as resp:
            text = await resp.text()
            soup = BeautifulSoup(text, 'html.parser')
            ul = soup.select_one('#content > div.left_cont > div > div.section.hk_news > div.section_cont > ul')
            if not ul:
                return []

            urls = []
            for a in ul.find_all('a', href=True):
                href = a['href']
                if '/article/' in href:
                    urls.append(href)
            return list(set(urls))  # 중복 제거
    except Exception as e:
        print(f"[get_article_urls error] {page_url} - {e}")
        return []

# ▶ 기사 상세 파싱 함수
async def fetch_article(session, url):
    try:
        async with session.get(url, headers=headers) as resp:
            html = await resp.text()
            soup = BeautifulSoup(html, 'html.parser')

            hostname = urlparse(url).hostname

            # ✅ 1. magazine.hankyung.com용 로직
            if 'magazine.hankyung.com' in hostname:
                return {
                    'header': soup.select_one('#contents h1.news-tit').text.strip() if soup.select_one('#contents h1.news-tit') else None,
                    'summary': None,
                    'content': soup.select_one('#magazineView').text.strip() if soup.select_one('#magazineView') else None,
                    'url': url,
                    'datetime': soup.select_one('#contents span.txt-num').text.strip() if soup.select_one('#contents span.txt-num') else None,
                }

            # ✅ 2. www.hankyung.com일 경우 기존 로직
            elif 'hankyung.com' in hostname:
                return {
                    'header': soup.select_one('h1.headline').text.strip() if soup.select_one('h1.headline') else None,
                    'summary': soup.select_one('div.summary').text.strip() if soup.select_one('div.summary') else None,
                    'content': soup.select_one('#articletxt').text.strip() if soup.select_one('#articletxt') else None,
                    'url': url,
                    'datetime': soup.select_one('div.datetime span.txt-date').text.strip() if soup.select_one('div.datetime span.txt-date') else None,
                }

            # ✅ 알 수 없는 도메인
            else:
                print(f"⚠️ 알 수 없는 호스트: {hostname}")
                return {'url': url, 'header': None, 'summary': None, 'content': None, 'datetime': None}

    except Exception as e:
        print(f"[fetch_article error] {url} - {e}")
        return {'url': url, 'header': None, 'summary': None, 'content': None, 'datetime': None}
# ▶ 메인 비동기 루프
async def extract_news_data_async(query_text, page_range):
    base_url = 'https://search.hankyung.com/search/news?query={query}&page={page}'
    search_urls = [base_url.format(query=query_text, page=p+1) for p in range(page_range)]

    async with aiohttp.ClientSession() as session:
        # 1. 페이지별 기사 링크 수집
        tasks = [get_article_urls(session, url) for url in search_urls]
        results = await asyncio.gather(*tasks)
        article_urls = list(set([url for sublist in results for url in sublist]))

        print(f"🔗 총 {len(article_urls)}개의 기사 URL 수집됨")

        # 2. 기사 본문 수집
        article_tasks = [fetch_article(session, url) for url in article_urls]
        articles = await asyncio.gather(*article_tasks)

        # 3. ticker 컬럼 추가
        for article in articles:
            article['ticker'] = query_text

        # 4. 비어 있으면 dummy row 추가
        if not articles:
            articles = [{
                'header': None,
                'summary': None,
                'content': None,
                'url': None,
                'datetime': None,
                'ticker': query_text
            }]
            print("⚠️ 수집된 기사가 없어 None 값으로 대체 저장합니다.")

        # 4. DuckDB 저장
        df = pd.DataFrame(articles)
        con = duckdb.connect('../DB/Customer_news.db')

        # Pandas DataFrame을 DuckDB에서 참조할 수 있도록 등록
        con.register('temp_df', df)

        # 테이블이 없다면 생성
        con.execute("""
            CREATE TABLE IF NOT EXISTS articles AS
            SELECT * FROM temp_df LIMIT 0
        """)

        # 데이터 삽입
        con.execute("INSERT INTO articles SELECT * FROM temp_df")

        # 정리
        con.unregister('temp_df')
        con.close()

        print(f"✅ 저장 완료: ../DB/Customer_news.db (ticker = {query_text})")

# ▶ 실행 함수
def extract_news_data(query_text, page_range):
    loop = asyncio.get_event_loop()
    loop.run_until_complete(extract_news_data_async(query_text, page_range))

In [78]:
for ticker in Customer_A['구성종목 종목명']:
    extract_news_data(ticker,50)

🔗 총 500개의 기사 URL 수집됨
✅ 저장 완료: ../DB/Customer_news.db (ticker = 우리금융지주)
🔗 총 500개의 기사 URL 수집됨
✅ 저장 완료: ../DB/Customer_news.db (ticker = 삼성화재)
🔗 총 500개의 기사 URL 수집됨
✅ 저장 완료: ../DB/Customer_news.db (ticker = 삼성생명)
🔗 총 500개의 기사 URL 수집됨
✅ 저장 완료: ../DB/Customer_news.db (ticker = 카카오뱅크)
🔗 총 500개의 기사 URL 수집됨
✅ 저장 완료: ../DB/Customer_news.db (ticker = 메리츠금융지주)
🔗 총 500개의 기사 URL 수집됨
✅ 저장 완료: ../DB/Customer_news.db (ticker = 현대모비스)
🔗 총 500개의 기사 URL 수집됨
✅ 저장 완료: ../DB/Customer_news.db (ticker = 현대오토에버)
🔗 총 500개의 기사 URL 수집됨
✅ 저장 완료: ../DB/Customer_news.db (ticker = SK하이닉스)
🔗 총 500개의 기사 URL 수집됨
✅ 저장 완료: ../DB/Customer_news.db (ticker = 현대글로비스)
🔗 총 500개의 기사 URL 수집됨
✅ 저장 완료: ../DB/Customer_news.db (ticker = 현대차)
🔗 총 500개의 기사 URL 수집됨
✅ 저장 완료: ../DB/Customer_news.db (ticker = SK하이닉스)
🔗 총 500개의 기사 URL 수집됨
✅ 저장 완료: ../DB/Customer_news.db (ticker = 삼성전자)
🔗 총 500개의 기사 URL 수집됨
✅ 저장 완료: ../DB/Customer_news.db (ticker = 한미반도체)
🔗 총 500개의 기사 URL 수집됨
✅ 저장 완료: ../DB/Customer_news.db (ticker = 파크시스템스)
🔗 총 500개의 기사 URL 

저장이 잘 되었는 지 확인

In [22]:
cus_news = duckdb.connect('../DB/Customer_news.db')

In [23]:
cusA_news_df = cus_news.execute('select * from articles').fetch_df()
cus_news.close()

In [24]:
cusA_news_df.head(5)

,header,summary,content,url,datetime,ticker
0,"18일, 거래소 외국인 순매수상위에 전기,전자 업종 4종목",None,"외국인 투자자는 18일 거래소에서 삼성전자, NAVER, 크래프톤 등을 중점적으로 ...",https://www.hankyung.com/article/202506184343L,2025.06.18 18:35,우리금융지주
1,"""실적·주주환원 훈풍""…은행·증권주 신고가 행진, 스탁론 매수세도 유입",None,국내 은행 및 증권주들이 2분기 실적 호조와 주주환원 기대감에 힘입어 강세를 이어가...,https://www.hankyung.com/article/202507096279a,2025.07.09 10:30,우리금융지주
2,"12일, 외국인 거래소에서 한화에어로스페이스(+5.3%), 현대차(+0.25%) 등...",None,"외국인 투자자는 12일 거래소에서 한화에어로스페이스, 현대차, 현대건설 등을 중점적...",https://www.hankyung.com/article/202506121791L,2025.06.12 18:35,우리금융지주
3,"금감원, 우리금융 경평 '2→3등급' 결론…이번주 통보할 듯",None,금감원 / 사진=노정동 기자\n\n 금융감독원이 우리금융...,https://www.hankyung.com/article/2025031784556,2025.03.17 11:18,우리금융지주
4,한도 초과 걱정 없이 연 4%대 금리로 저점 집중투자 시작하기!,None,부자네스탁론이 특별 이벤트로 5년고정 연 4.9%의 저금리 스탁론 상품을 출시하면서...,https://www.hankyung.com/article/202506056123a,2025.06.05 14:38,우리금융지주


In [25]:
cusA_news_df.shape

(10000, 6)

## 번외) 날짜 전처리 확인 : 최근 N일치 가져오는 로직 생성

In [26]:
cusA_news_df['Date'] = cusA_news_df.datetime.str[:10]
cusA_news_df = cusA_news_df.drop('datetime',axis=1)

In [54]:
cusA_news_df['Date'] = pd.to_datetime(cusA_news_df['Date'])
# 2. 기준일 계산 (오늘 날짜 - 5일)
today = pd.to_datetime("2025-07-30")
five_days_ago = today - timedelta(days=30)

# 3. 최근 5일치 필터링
recent_news_df = cusA_news_df[cusA_news_df['Date'] >= five_days_ago]

In [55]:
recent_news_df

,header,summary,content,url,ticker,Date
1,"""실적·주주환원 훈풍""…은행·증권주 신고가 행진, 스탁론 매수세도 유입",None,국내 은행 및 증권주들이 2분기 실적 호조와 주주환원 기대감에 힘입어 강세를 이어가...,https://www.hankyung.com/article/202507096279a,우리금융지주,2025-07-09
5,"미래에셋만 너무 비싸다?…""PBR 1.2배도 가능""",None,영상 모듈 닫기\n\n\n\n\n<앵커> 증시 상승세에 힘입어 국내 증권사들도 2분...,https://www.hankyung.com/article/2025071501385,우리금융지주,2025-07-15
7,"09일, 외국인 거래소에서 삼성전자(-1.63%), 두산에너빌리티(-3.3%) 등 순매도",None,"외국인 투자자는 09일 거래소에서 삼성전자, 두산에너빌리티, 삼성SDI 등을 중점적...",https://www.hankyung.com/article/202507098435L,우리금융지주,2025-07-09
8,"01일, 코스닥 외국인 순매수상위에 일반전기전자 업종 5종목",None,"외국인 투자자는 01일 코스닥에서 리가켐바이오, 솔브레인, 디앤디파마텍 등을 중점적...",https://www.hankyung.com/article/202508017025L,우리금융지주,2025-08-01
13,반도체 대장주 자리 꿰차더니…SK하이닉스 개미들 '두근두근',2분기 실적 시즌 돌입…영업이익 추정치 살펴보니\n\n하이닉스 영업익 첫 9조 넘을...,2분기 실적 발표 시즌에 본격 돌입하면서 주도주의 성적표가 윤곽을 드러내고 있다. ...,https://www.hankyung.com/article/2025072252001,우리금융지주,2025-07-22
...,...,...,...,...,...,...
9995,"코스피, 3200선 회복…테슬라 업은 삼성전자, 7만원대 탈환",코스닥은 0.3% '하락'\n원·달러 환율 1382원에 주간거래 마쳐,28일 서울 중구 하나은행 본점 딜링룸에서 직원들이 업무를 보고 있다. /사진=연합...,https://www.hankyung.com/article/2025072866196,LG에너지솔루션,2025-07-28
9996,"코스피, 3년10개월 만에 3200선 돌파…'연고점 또 경신'",삼성전자·하이닉스 2%대 강세,사진=연합뉴스\n\n 코스피지수가 11일 개인투자자의 매...,https://www.hankyung.com/article/2025071119236,LG에너지솔루션,2025-07-11
9997,"코스피, 3190선 약세 출발…코스닥은 강보합",None,전날인 14일 오후 서울 중구 하나은행 본점 딜링룸 전광판. /사진=뉴스1\n\n ...,https://www.hankyung.com/article/2025071582926,LG에너지솔루션,2025-07-15
9998,"이자 부담은 최소로, 투자 효율은 최대로! 신용대출 3%대 활용법",None,"전송종목 : 바이오비쥬, 엘브이엠씨홀딩스, 크리스탈신소재, GRT, 잉글우드랩최근 ...",https://www.hankyung.com/article/202507071499a,LG에너지솔루션,2025-07-07


In [56]:
recent_news_df= recent_news_df.drop_duplicates()

In [ ]:
# 최근 30일치 뉴스 기사 필터링 시 남아있는 뉴스 기사 개수
recent_news_df.groupby('ticker').count()

,header,summary,content,url,Date
ticker,,,,,
DB하이텍,30,4,30,30,30
LG에너지솔루션,500,96,500,500,500
POSCO홀딩스,464,23,464,464,464
SK하이닉스,500,119,500,500,500
기아,411,191,411,411,411
메리츠금융지주,51,5,51,51,51
삼성생명,140,45,140,140,140
삼성전자,500,161,500,500,500
삼성화재,96,31,96,96,96


## 질문 생성 (For Labeling)  -- 주말 작업 예정 .. 모델 성능평가를 위함
Part_1 : Re-Ranking 라벨링

In [70]:
news_test = recent_news_df.iloc[1].content

In [ ]:
# 2. 프롬프트 템플릿 정의
template = """
너는 투자 뉴스 기사를 읽고, 사람들이 이 기사를 보고 물어볼 법한 질문을 만들어주는 AI야.
뉴스 본문을 보고 이 기사의 핵심을 파악한 질문을 아래 예시를 보고 **한 문장**으로 생성해줘.
예시 
: POSCO홀딩스는 배당소득 분리과세 혜택을 받을 수 있나요?

검색 종목 : 
{ticker_name}

뉴스 본문:
{news_content}

질문:"""

prompt = ChatPromptTemplate.from_template(template)

# 3. 체인 구성
chain = prompt | llm | StrOutputParser()

# 4. 실행 예시
news = news_test
query = chain.invoke({"news_content": news,'ticker_name' : '우리금융지주'})
print("생성된 질문:", query)

생성된 질문: 미래에셋증권의 PBR이 어떤 수준을 넘어가면 추가 상승여력이 낮다는 평가를 받을까요?


## 6. VectorDB 저장

In [18]:
# 4. OpenAI 임베딩 모델 로딩
embedding = OpenAIEmbeddings(model="text-embedding-3-large")

persist_directory = "../VectorDB/chroma_news_db"

vectordb = Chroma(
    persist_directory=persist_directory,
    embedding_function=embedding
)

In [32]:
def pick_splitter_by_length(text_len: int) -> RecursiveCharacterTextSplitter:
    """
    뉴스 본문의 길이에 따라 적절한 텍스트 분할기를 반환합니다.
    """
    if text_len <= 1200:
        # 짧은 기사 → 굳이 자르지 않고 1덩어리로 처리
        return RecursiveCharacterTextSplitter(
            chunk_size=1200,
            chunk_overlap=0,
            separators=["\n\n", "\n", " ", ""]
        )
    elif text_len <= 10_000:
        # 중간 길이 → 일반적인 1,200자 기준으로 분할
        return RecursiveCharacterTextSplitter(
            chunk_size=1200,
            chunk_overlap=150,
            separators=["\n\n", "\n", " ", ""]
        )
    elif text_len <= 50_000:
        # 긴 기사 → 덩어리를 좀 더 키움
        return RecursiveCharacterTextSplitter(
            chunk_size=1800,
            chunk_overlap=200,
            separators=["\n\n", "\n", " ", ""]
        )
    else:
        # 초장문 → 더 크게 자르되, 요약도 고려 (이건 후속 처리 필요)
        return RecursiveCharacterTextSplitter(
            chunk_size=2000,
            chunk_overlap=200,
            separators=["\n\n", "\n", " ", ""]
        )
    

In [33]:
# 2. 문서 리스트 생성 (chunk + metadata 포함)
def make_documents(df):
    docs = []

    for idx, row in df.iterrows():
        text = row["content"]
        splitter = pick_splitter_by_length(len(text))
        chunks = splitter.split_text(text)

        for i, chunk in enumerate(chunks):
            metadata = {
                "title": row["header"],
                "url": row["url"],
                "Date": row["Date"],
                "ticker": row.get("ticker", "None"),
                "chunk_idx": i,
                "original_idx": idx,
            }
            docs.append(Document(page_content=chunk, metadata=metadata))

    return docs


In [34]:
def get_recent_articles(df: pd.DataFrame, ticker: str, days: int = 5):
    df['Date'] = pd.to_datetime(df['Date'])
    today = df['Date'].max()
    recent_df = df[
        (df['ticker'] == ticker) &
        (df['Date'] >= today - timedelta(days=days))
    ]
    return recent_df.sort_values(by="Date", ascending=False)


In [40]:
def make_doc_id(d: Document) -> str:
    """
    url + chunk_idx(없으면 0) 조합으로 안정적인 고유 id 생성
    """
    base = f"{d.metadata.get('url','')}_{d.metadata.get('chunk_idx', 0)}"
    return md5(base.encode("utf-8")).hexdigest()

def chunks(lst, size):
    for i in range(0, len(lst), size):
        yield lst[i:i + size]


In [ ]:
recent_df_woori = get_recent_articles(cusA_news_df,ticker='우리금융지주',days = 30)
recent_df_woori['Date'] = recent_df_woori.Date.astype('str')
recent_df_woori_docs = make_documents(recent_df_woori)
BATCH = 64  # 상황에 맞게 조절

for docs in tqdm(chunks(recent_df_woori_docs, BATCH), total=(len(recent_df_woori_docs) + BATCH - 1) // BATCH):

    ids = [make_doc_id(d) for d in docs]
    vectordb.add_documents(documents=docs, ids=ids)

In [43]:
len(recent_df_woori_docs)

219

In [41]:
BATCH = 64  # 상황에 맞게 조절

for docs in tqdm(chunks(recent_df_woori_docs, BATCH), total=(len(recent_df_woori_docs) + BATCH - 1) // BATCH):

    ids = [make_doc_id(d) for d in docs]
    vectordb.add_documents(documents=docs, ids=ids)

  0%|          | 0/4 [00:00<?, ?it/s]

100%|██████████| 4/4 [00:11<00:00,  2.89s/it]


In [42]:
print("Number of documents in DB:", vectordb._collection.count())

Number of documents in DB: 219


## 7. cross-encoder 모델(w/langchain 예시)

Re-ranker 사용하기 전

In [19]:
retriever = vectordb.as_retriever(search_kwargs={"k": 10,'filter' : {'ticker' : '우리금융지주'}})

In [20]:
def pretty_print_docs(docs):
    print(
        f"\n{'-' * 100}\n".join(
            [f"Document {i + 1}:\n\n" + d.page_content for i, d in enumerate(docs)]
        )
    )

In [21]:
query = '''
이 뉴스들 중에서 "우리금융지주"가 핵심 주제로 다뤄진 기사만 알려줘.
다른 회사 언급이 많거나, 우리금융지주가 단순히 함께 언급된 기사라면 제외해줘.
'''
docs = retriever.invoke(query)
pretty_print_docs(docs)

Document 1:

<앵커> 오늘 금융지주 주가가 오전에 잠시 조정받는 듯했는데 KB금융 제외하곤 낙폭을 줄여가는 모습입니다. 앞서 최민정 기자 언급한 자사주 의무 소각 관련해 금융지주들에는 어떤 영향이 있을지 경제부 유주안 기자와 이야기 나눠봅니다. 금융지주들은 자사주 매입과 소각에 꽤 적극적인데, 보유중인 자사주의 비중은 어떻습니까?<기자>4대 금융지주 자사주 보유 현황(KB 4.54%, 신한 2.09%, 하나 3.77%, 우리 1.16%)은, 보시다시피 비중이 크다고 보이진 않습니다.자사주 소각 의무화 추진에 따른 수혜주로 꼽히는 미래에셋(22.98) 대신(25.12) 신영(52.58%) 등에 비하면 매우 작은 수준이고요, 일반지주사들(티와이홀딩스 29.79%, SK 24.8%, 롯데지주 27.37% 등)에 비해서도 비중이 낮습니다.과거 역사를 살펴보면 금융주에 있어서는 자사주 ‘지배의 도구’보다는 인수합병 때 주식교환을 하기 위해, 전략적 투자자와 주식을 스왑하는 차원에서 일정 부분 보유해왔던 것을 알 수 있고요, 최근 들어서는 매입하면 바로 소각해서 주식의 가치를 올리는, 주주환원의 수단으로 자사주를 활용해 왔습니다.따라서, 자사주 소각 의무화에 따른 강제성 측면에서 금융지주에 미치는 단기적 직접적 영향은 크지 않겠습니다. 중장기적으로 볼 때 금융지주의 주주환원에 보다 힘이 실릴 것이란 기대감으로 이어질 수 있는데요,정책이 추구하는 목표를 이미 금융지주사들이 실행해 옮기고 있는 것이고, 향후 상장사들 사이에 자사주 매입소각이 보편화되면 재무적 여력과, 주주환원에 대한 강한 의지를 가진 금융사들이 더 적극적인 주주환원을 펼칠 수 있는 배경이 될 수 있을 것입니다.또한, 정부가 함께 추진중인 배당소득 분리과세 정책은 배당성향이 높고, 분기배당에 적극적인 금융주에게 특히 긍정적인 정책으로 받아들여지고 있습니다.<앵커> 어제 삼성전자 실적으로 인해 다소 가라앉은 분위기로 어닝시즌 개막되었는데요. 금융주 실적은 어떨 걸로 전망됩니까?<기자> 4대 지주만 놓

In [22]:
docs[0].metadata

{'original_idx': 245,
 'url': 'https://www.hankyung.com/article/2025070981175',
 'Date': '2025-07-09',
 'title': '더 커진 주주환원 기대감…금융주, 어닝시즌 관전포인트',
 'chunk_idx': 1,
 'ticker': '우리금융지주'}

Re-ranker 모델 사용 후

In [ ]:
#model = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-base")
model = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-v2-m3")
compressor = CrossEncoderReranker(model=model, top_n=10)
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=retriever
)

/Users/Work/SU/Project/streamlit/Streamlit/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [24]:
CrossEncoder_prompt = '''
이 뉴스들 중에서 "우리금융지주"가 핵심 주제로 다뤄진 기사만 알려줘.
다른 회사 언급이 많거나, 우리금융지주가 단순히 함께 언급된 기사라면 제외해줘.
'''

In [25]:
compressed_docs = compression_retriever.invoke(CrossEncoder_prompt)
pretty_print_docs(compressed_docs)

/Users/Work/SU/Project/streamlit/Streamlit/lib/python3.13/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `XLMRobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


Document 1:

◆ 주체별 매매동향- 전일 외국인 대량 순매수지난 한달을 기준으로 보면 외국인이 296.2만주를 순매수했고, 개인들도 41.8만주를 순매수했다. 하지만 기관은 294.9만주를 순매도했다. 같은 기간 이 종목의 거래 비중은 외국인과 개인이 각각 51.3%, 26.2%로 비중이 높다.더욱이 전일 외국인이 대량 순매수를 하고 있어 투자자들의 관심이 집중되고 있다.[그래프]우리금융지주 외국인/기관 매매동향











◆ 최근 애널리스트 분석의견- 갈수록 돋보일 고배당 매력 - NH투자증권, BUY07월 09일 NH투자증권의 정준섭 애널리스트는 우리금융지주에 대해 "'25년 2분기 지배순이익은 다소 부진할 예정. 경상 실적(이자이 익, 비이자이익)은 견조하나, 책준형 신탁 충당금 등으로 대손비용률 상승 (65bp, +20bp q-q, +21bp y-y)이 예상되기 때문. 우리금융지주는 하반기 자사주 매입 기대감은 낮지만, 배당 매력은 갈수록 돋보일 전망"이라고 분석하며, 투자의견 'BUY', 목표주가 '29,000원'을 제시했다.한경로보뉴스이 기사는 한국경제신문과 금융 AI 전문기업 씽크풀이 공동 개발한 기사 자동생성 알고리즘에 의해 실시간으로 작성된 것입니다.
----------------------------------------------------------------------------------------------------
Document 2:

이미지 크게보기



                15일 금융정보업체 에프앤가이드에 따르면 4대 금융지주의 지난 2분기 순이익 추정치는 총 5조80억원이다. 작년 2분기(5조1241억원)와 비교해 1161억원(2.3%) 줄어든 규모다.4대 금융지주의 분기 단위 순이익이 전년 동기 대비 감소한 것은 작년 1분기(-6910억원) 이후 처음이다. 작년 1분기엔 4대 시중은행(국민·신한·하나·우리)이 홍콩 H지수 ELS 배상을 위해 총 1조3174억원을 일회성 비용인 충당부채로 적립한 

In [26]:
compressed_docs[0]

Document(id='9c0f5a36f30a569d0c7d646d8d096466', metadata={'original_idx': 237, 'Date': '2025-07-14', 'title': "'우리금융지주' 52주 신고가 경신, 전일 외국인 대량 순매수", 'ticker': '우리금융지주', 'url': 'https://www.hankyung.com/article/202507145874L', 'chunk_idx': 0}, page_content='◆ 주체별 매매동향- 전일 외국인 대량 순매수지난 한달을 기준으로 보면 외국인이 296.2만주를 순매수했고, 개인들도 41.8만주를 순매수했다. 하지만 기관은 294.9만주를 순매도했다. 같은 기간 이 종목의 거래 비중은 외국인과 개인이 각각 51.3%, 26.2%로 비중이 높다.더욱이 전일 외국인이 대량 순매수를 하고 있어 투자자들의 관심이 집중되고 있다.[그래프]우리금융지주 외국인/기관 매매동향\n\n\n\n\n\n\n\n\n\n\n\n◆ 최근 애널리스트 분석의견- 갈수록 돋보일 고배당 매력 - NH투자증권, BUY07월 09일 NH투자증권의 정준섭 애널리스트는 우리금융지주에 대해 "\'25년 2분기 지배순이익은 다소 부진할 예정. 경상 실적(이자이 익, 비이자이익)은 견조하나, 책준형 신탁 충당금 등으로 대손비용률 상승 (65bp, +20bp q-q, +21bp y-y)이 예상되기 때문. 우리금융지주는 하반기 자사주 매입 기대감은 낮지만, 배당 매력은 갈수록 돋보일 전망"이라고 분석하며, 투자의견 \'BUY\', 목표주가 \'29,000원\'을 제시했다.한경로보뉴스이 기사는 한국경제신문과 금융 AI 전문기업 씽크풀이 공동 개발한 기사 자동생성 알고리즘에 의해 실시간으로 작성된 것입니다.')

검색 유사도 측정

In [65]:
raw_docs = retriever.get_relevant_documents(CrossEncoder_prompt)
pairs = [(CrossEncoder_prompt, d.page_content) for d in raw_docs]
scores = model.score(pairs)

/var/folders/xq/zzsj9f116r7brgm2wm610nx00000gn/T/ipykernel_66795/2960513470.py:1: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  raw_docs = retriever.get_relevant_documents(CrossEncoder_prompt)
/Users/Work/SU/Project/streamlit/Streamlit/lib/python3.13/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `XLMRobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


In [69]:
# 3) 점수 붙이고 재정렬
scored_docs = []
for d, s in zip(raw_docs, scores):
    dd = deepcopy(d)
    dd.metadata["relevance_score"] = float(s)
    scored_docs.append(dd)


In [70]:
scored_docs.sort(key=lambda x: x.metadata["relevance_score"], reverse=True)

In [71]:
# 4) 확인 - raw data
for i, d in enumerate(scored_docs[:50], 1):
    print(f"{i:02d} | {d.metadata['relevance_score']:.4f} | {d.metadata.get('title')}")

01 | 0.6022 | '우리금융지주' 52주 신고가 경신, 전일 외국인 대량 순매수
02 | 0.3970 | "우리금융지주, 연말로 갈수록 고배당 부각…목표가↑"-NH
03 | 0.3583 | 대출 수익성 악화에…4대 금융 실적 꺾였다
04 | 0.2479 | '우리금융지주' 52주 신고가 경신, 갈수록 돋보일 고배당 매력 - NH투자증권, BUY
05 | 0.2153 | 4대금융 2분기 순익 5.4조…사상 최대 실적
06 | 0.1130 | 더 커진 주주환원 기대감…금융주, 어닝시즌 관전포인트
07 | 0.1089 | "실적·주주환원 훈풍"…은행·증권주 신고가 행진, 스탁론 매수세도 유입
08 | 0.0163 | 금융지주 영구채 '큰장' 선다…신한·하나 등 1.8조원 쏟아질 듯
09 | 0.0100 | 대출 수익성 악화에…4대 금융 실적 꺾였다
10 | 0.0047 | 더 커진 주주환원 기대감…금융주, 어닝시즌 관전포인트


In [72]:
reranked_docs = compressor.compress_documents(raw_docs, query=CrossEncoder_prompt)

/Users/Work/SU/Project/streamlit/Streamlit/lib/python3.13/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `XLMRobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


In [73]:
rerank_model_results_title = list(map(lambda x : x.metadata['title'],reranked_docs))

In [75]:
for i, d in enumerate(rerank_model_results_title[:50], 1):
    print(f"{i:02d} | {d}")

01 | '우리금융지주' 52주 신고가 경신, 전일 외국인 대량 순매수
02 | "우리금융지주, 연말로 갈수록 고배당 부각…목표가↑"-NH
03 | 대출 수익성 악화에…4대 금융 실적 꺾였다
04 | '우리금융지주' 52주 신고가 경신, 갈수록 돋보일 고배당 매력 - NH투자증권, BUY
05 | 4대금융 2분기 순익 5.4조…사상 최대 실적
06 | 더 커진 주주환원 기대감…금융주, 어닝시즌 관전포인트
07 | "실적·주주환원 훈풍"…은행·증권주 신고가 행진, 스탁론 매수세도 유입
08 | 금융지주 영구채 '큰장' 선다…신한·하나 등 1.8조원 쏟아질 듯
09 | 대출 수익성 악화에…4대 금융 실적 꺾였다
10 | 더 커진 주주환원 기대감…금융주, 어닝시즌 관전포인트


### original 본문 가져오기

In [27]:
def get_full_article_from_chroma(original_idx: int, kind : str, vectordb) -> dict:
    """original_idx 기준으로 chunk들을 모아 원문 복원"""
    # 1. 해당 article의 모든 chunk 가져오기
    VecDB = vectordb._collection
    results = VecDB.get(
      where = { "$and" : [ # 빈 쿼리로 전체 탐색
            {"original_idx": original_idx}, 
            {"ticker": "우리금융지주"}
            ]
         },
         include = ['documents','metadatas']   
        )
        
    if not results:
        return {"error": f"No chunks found for original_idx {original_idx}"}

    #print(results)
    #print(results['metadatas'][0]['chunk_idx'])

    # 4. 대표 metadata 하나 뽑아 저장
    return {
        "title": results['metadatas'][0]["title"],
        "url": results['metadatas'][0]["url"],
        "Date": results['metadatas'][0]["Date"],
        "ticker": results['metadatas'][0]["ticker"],
        "content": results['documents'][0]
    }

In [28]:
original_idxs = list(map(lambda x: x.metadata['original_idx'], compressed_docs))

In [29]:
original_idxs

[237, 158, 23, 475, 245, 1, 229, 158, 245, 258]

In [30]:
get_full_article_from_chroma(237, kind='우리금융지주',vectordb=vectordb)

{'title': "'우리금융지주' 52주 신고가 경신, 전일 외국인 대량 순매수",
 'url': 'https://www.hankyung.com/article/202507145874L',
 'Date': '2025-07-14',
 'ticker': '우리금융지주',
 'content': '◆ 주체별 매매동향- 전일 외국인 대량 순매수지난 한달을 기준으로 보면 외국인이 296.2만주를 순매수했고, 개인들도 41.8만주를 순매수했다. 하지만 기관은 294.9만주를 순매도했다. 같은 기간 이 종목의 거래 비중은 외국인과 개인이 각각 51.3%, 26.2%로 비중이 높다.더욱이 전일 외국인이 대량 순매수를 하고 있어 투자자들의 관심이 집중되고 있다.[그래프]우리금융지주 외국인/기관 매매동향\n\n\n\n\n\n\n\n\n\n\n\n◆ 최근 애널리스트 분석의견- 갈수록 돋보일 고배당 매력 - NH투자증권, BUY07월 09일 NH투자증권의 정준섭 애널리스트는 우리금융지주에 대해 "\'25년 2분기 지배순이익은 다소 부진할 예정. 경상 실적(이자이 익, 비이자이익)은 견조하나, 책준형 신탁 충당금 등으로 대손비용률 상승 (65bp, +20bp q-q, +21bp y-y)이 예상되기 때문. 우리금융지주는 하반기 자사주 매입 기대감은 낮지만, 배당 매력은 갈수록 돋보일 전망"이라고 분석하며, 투자의견 \'BUY\', 목표주가 \'29,000원\'을 제시했다.한경로보뉴스이 기사는 한국경제신문과 금융 AI 전문기업 씽크풀이 공동 개발한 기사 자동생성 알고리즘에 의해 실시간으로 작성된 것입니다.'}

In [31]:
top_n_original = [get_full_article_from_chroma(idx,kind='우리금융지주',vectordb=vectordb) for idx in original_idxs]

In [32]:
top_n_original[0]

{'title': "'우리금융지주' 52주 신고가 경신, 전일 외국인 대량 순매수",
 'url': 'https://www.hankyung.com/article/202507145874L',
 'Date': '2025-07-14',
 'ticker': '우리금융지주',
 'content': '◆ 주체별 매매동향- 전일 외국인 대량 순매수지난 한달을 기준으로 보면 외국인이 296.2만주를 순매수했고, 개인들도 41.8만주를 순매수했다. 하지만 기관은 294.9만주를 순매도했다. 같은 기간 이 종목의 거래 비중은 외국인과 개인이 각각 51.3%, 26.2%로 비중이 높다.더욱이 전일 외국인이 대량 순매수를 하고 있어 투자자들의 관심이 집중되고 있다.[그래프]우리금융지주 외국인/기관 매매동향\n\n\n\n\n\n\n\n\n\n\n\n◆ 최근 애널리스트 분석의견- 갈수록 돋보일 고배당 매력 - NH투자증권, BUY07월 09일 NH투자증권의 정준섭 애널리스트는 우리금융지주에 대해 "\'25년 2분기 지배순이익은 다소 부진할 예정. 경상 실적(이자이 익, 비이자이익)은 견조하나, 책준형 신탁 충당금 등으로 대손비용률 상승 (65bp, +20bp q-q, +21bp y-y)이 예상되기 때문. 우리금융지주는 하반기 자사주 매입 기대감은 낮지만, 배당 매력은 갈수록 돋보일 전망"이라고 분석하며, 투자의견 \'BUY\', 목표주가 \'29,000원\'을 제시했다.한경로보뉴스이 기사는 한국경제신문과 금융 AI 전문기업 씽크풀이 공동 개발한 기사 자동생성 알고리즘에 의해 실시간으로 작성된 것입니다.'}

In [33]:
original_contents = list(map(lambda x: x['content'],top_n_original))

In [34]:
total_contents = ''.join(original_contents)

In [39]:
total_contents

'◆ 주체별 매매동향- 전일 외국인 대량 순매수지난 한달을 기준으로 보면 외국인이 296.2만주를 순매수했고, 개인들도 41.8만주를 순매수했다. 하지만 기관은 294.9만주를 순매도했다. 같은 기간 이 종목의 거래 비중은 외국인과 개인이 각각 51.3%, 26.2%로 비중이 높다.더욱이 전일 외국인이 대량 순매수를 하고 있어 투자자들의 관심이 집중되고 있다.[그래프]우리금융지주 외국인/기관 매매동향\n\n\n\n\n\n\n\n\n\n\n\n◆ 최근 애널리스트 분석의견- 갈수록 돋보일 고배당 매력 - NH투자증권, BUY07월 09일 NH투자증권의 정준섭 애널리스트는 우리금융지주에 대해 "\'25년 2분기 지배순이익은 다소 부진할 예정. 경상 실적(이자이 익, 비이자이익)은 견조하나, 책준형 신탁 충당금 등으로 대손비용률 상승 (65bp, +20bp q-q, +21bp y-y)이 예상되기 때문. 우리금융지주는 하반기 자사주 매입 기대감은 낮지만, 배당 매력은 갈수록 돋보일 전망"이라고 분석하며, 투자의견 \'BUY\', 목표주가 \'29,000원\'을 제시했다.한경로보뉴스이 기사는 한국경제신문과 금융 AI 전문기업 씽크풀이 공동 개발한 기사 자동생성 알고리즘에 의해 실시간으로 작성된 것입니다.KB 신한 하나 우리 등 4대 금융지주의 올해 2분기 합산 순이익이 전년 동기 대비 감소한 것으로 파악됐다. 금융지주의 실적이 전년 동기 대비 감소한 것은 홍콩 H지수 주가연계증권(ELS) 배상으로 1조원 넘는 일회성 비용이 발생한 2024년 1분기를 제외하면 1년 반 만에 처음이다. 금융회사의 핵심 수익원인 이자수익이 줄줄이 감소한 결과다. 금융지주의 핵심 자회사인 은행들이 가계대출 억제 정책과 경기 침체로 대출 자산을 확대하기 어려운 만큼 향후 금융지주의 실적 감소세가 본격화할 것이란 관측이 제기된다.\n                    \n\n\n\n\n\n\n이미지 크게보기◆ 최근 애널리스트 분석의견- 갈수록 돋보일 고배당 매력 - NH투자증권, BUY07월 09

## 요약하기

### 요약함수 호출
- 가져온 원 본문을 전부 적용하기

In [45]:
from summart_function_openai_2 import NewsSummaryAgent, build_summary_graph  # 너가 만든 것

In [54]:
def summarize_top_articles_2(total_contents: str,ticker:str) -> pd.DataFrame:
    agent = NewsSummaryAgent()
    runnable = build_summary_graph(agent)

    rows = []
    doc = total_contents

    state = {"article": doc, "summary": "", "feedback": "", "iteration": 0}
    result = runnable.invoke(state)

    print("✅ 실행 결과 키:", result.keys())
    # 여기서 final_summary 반드시 존재해야 함(위 패치 기준)
    final_summary = result.get("final_summary")
    if not final_summary:
        print("❌ final_summary 없음. 디버그용 전체 상태:", result)
        # 계속 진행할지, 실패로 표기할지 선택

    rows.append({
        "ticker": ticker,
        "date": '2025-07-30', # 위 조회 기준일자로 연동시켜서 바꿀 예정
        "summary": final_summary,
        "feedback": result.get("last_feedback", "피드백 없음"),
    })

    return pd.DataFrame(rows)


In [56]:
result_df = summarize_top_articles_2(total_contents,ticker='우리금융지주')

summart : ✅ 주요 요약
- 외국인 투자자, 우리금융지주 대량 순매수
  외국인 투자자들이 최근 우리금융지주 주식을 대량으로 순매수하며, 이로 인해 투자자들의 관심이 집중되고 있음.

- 4대 금융지주, 비이자이익 증가로 2분기 순이익 증가
  KB, 신한, 하나, 우리 등 4대 금융지주의 2분기 순이익이 비이자이익 증가 덕분에 전년 동기 대비 증가했으며, 이는 환율 안정과 수수료 수입 확대에 기인함.

- 금융지주, 하반기 신종자본증권 발행 계획
  신한, 하나, 우리, 농협금융 등 주요 금융지주들이 하반기 신종자본증권 발행을 통해 자본 확충 및 유동성 확보를 계획하고 있음.

🔑 키워드
- 외국인 순매수
- 우리금융지주
- 4대 금융지주
- 비이자이익
- 신종자본증권
- 자본 확충
- 환율 안정
- 주주환원 정책
[should_stop] Iteration: 0
[should_stop] Feedback:
 - 정확성: 좋음
- 포괄성: 부족함 <reason> [요약에서 외국인 투자자의 대량 순매수에 대한 정보는 포함되었으나, 기관의 매도와 개인의 매수에 대한 정보가 누락되었습니다. 또한, 4대 금융지주의 순이익 증가에 대한 구체적인 수치나 분석 내용이 부족합니다.] </reason>
- 간결성: 좋음
- 문장구성: 좋음

**피드백:**
요약의 포괄성을 개선하기 위해, 기관의 매도와 개인의 매수에 대한 정보도 포함하는 것이 좋습니다. 또한, 4대 금융지주의 순이익 증가에 대한 구체적인 수치나 분석 내용을 추가하여 독자가 더 명확한 이해를 할 수 있도록 보완하면 좋겠습니다. 예를 들어, 각 금융지주의 순이익 증가율이나 비이자이익의 구체적인 증가율 등을 포함하면 더욱 포괄적인 요약이 될 것입니다.
[should_stop] next_step = no
====== result : refined_summary='✅ **개선된 요약**\n\n- **외국인 및 개인 투자자 매수, 기관 매도**  \n  최근 한 달 동안 외국인 투자자들이 우리금융지주 주식을 296

In [63]:
print(result_df.summary.values[0])

✅ **개선된 요약**

- **외국인 및 개인 투자자 매수, 기관 매도**  
  최근 한 달 동안 외국인 투자자들이 우리금융지주 주식을 296.2만 주 순매수했으며, 개인 투자자들도 41.8만 주를 순매수했습니다. 반면, 기관 투자자들은 294.9만 주를 순매도했습니다. 외국인과 개인의 거래 비중은 각각 51.3%와 26.2%로 높습니다.

- **4대 금융지주, 비이자이익 증가로 2분기 순이익 증가**  
  KB, 신한, 하나, 우리 등 4대 금융지주의 2분기 순이익은 비이자이익 증가 덕분에 전년 동기 대비 5.3% 증가한 5조3954억 원을 기록했습니다. 이는 환율 안정과 수수료 수입 확대에 기인합니다. 특히, 하나금융의 순이익은 13.4% 증가하며 가장 큰 폭의 성장을 보였습니다.

- **금융지주, 하반기 신종자본증권 발행 계획**  
  신한, 하나, 우리, 농협금융 등 주요 금융지주들이 하반기에 최대 1조7900억 원 규모의 신종자본증권을 발행하여 자본 확충 및 유동성 확보를 계획하고 있습니다. 이는 기존 신종자본증권의 차환 물량을 대비하기 위한 조치입니다.

🔑 **키워드**
- 외국인 순매수
- 개인 순매수
- 기관 순매도
- 우리금융지주
- 4대 금융지주
- 비이자이익
- 신종자본증권
- 자본 확충
- 환율 안정
- 주주환원 정책


# 과거 버전

In [204]:
import pandas as pd
from datetime import datetime, timedelta
# cross-encoder
from sentence_transformers import CrossEncoder
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

from summary_function_openai import NewsSummaryAgent, build_summary_graph  # 너가 만든 것

# 1. 모델 준비 (CrossEncoder for Re-ranking) # 예시 모델 하나 생성
rerank_model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# 2. 최근 5일치 필터링 함수
def get_recent_articles(df: pd.DataFrame, ticker: str, days: int = 5):
    df['Date'] = pd.to_datetime(df['Date'])
    today = df['Date'].max()
    recent_df = df[
        (df['ticker'] == ticker) &
        (df['Date'] >= today - timedelta(days=days))
    ]
    return recent_df.sort_values(by="Date", ascending=False)

# 3. huggingface 모델 이용.
def rerank_articles(df: pd.DataFrame,ticker:str ,query: str, top_k: int = 5):
    task_df= df[df.ticker == ticker]
    docs = task_df['content'].tolist()
    # 모델 이용?? 
    pairs = [(query, doc) for doc in docs]
    scores = rerank_model.predict(pairs)
    
    task_df_2 = task_df.copy()
    task_df_2['score'] = scores
    return task_df_2.sort_values(by='score', ascending=False).head(top_k)

# 4. 전체 요약 실행 함수
def summarize_top_articles(df: pd.DataFrame, ticker: str, query: str, top_k: int = 5):
    recent_df = get_recent_articles(df, ticker)
    top_df = rerank_articles(recent_df, query=query,ticker=ticker, top_k=top_k)

    agent = NewsSummaryAgent()
    runnable = build_summary_graph(agent)

    results = []
    for _, row in top_df.iterrows():
        state = {
            "article": row['content'],
            "summary": "",
            "feedback": "",
            "iteration": 0
        }
        result = runnable.invoke(state)


        print("✅ 실행 결과 타입:", type(result))
        print("✅ 실행 결과 키 목록:", result.keys())
        print("✅ 실행 결과 전체 내용:", result)

        if "final_summary" not in result:
            print("❌ final_summary 키가 없습니다. 중단합니다.")
            continue  # 또는 raise Exception("final_summary 없음")

        print(f'실행 결과 : {result}')
        results.append({
            "ticker": row['ticker'],
            "date": row['Date'],
            "header": row['header'],
            "url": row['url'],
            "summary": result["final_summary"], 
             "feedback": result.get("last_feedback", "피드백 없음")
        })
    return pd.DataFrame(results)


In [55]:
query = "우리금융지주 관련 시황"
ticker = "우리금융지주"  # 예시
result_df = summarize_top_articles(cusA_news_df, ticker=ticker, query=query, top_k=3)

/Users/Work/SU/Project/streamlit/Streamlit/lib/python3.13/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


[should_stop] Iteration: 1
[should_stop] Feedback:
 - 정확성: 좋음 <reason> 원문의 내용을 정확하게 반영하고 있음. </reason>
- 포괄성: 부족함 <reason> 원문에서 언급된 '기술 경쟁력 확보, 고객사 확대, 수익구조 개선 등 실질적 사업성과로 뒷받침돼야 한다는 지적' 등의 중요한 내용이 누락되었음. </reason>
- 간결성: 좋음 <reason> 불필요한 표현 없이 요약 내용을 간결하게 전달하였음. </reason>
- 문장구성: 좋음 <reason> 문장이 자연스럽고 명확하게 구성되어 있음. </reason>

[피드백]
요약의 포괄성이 부족한 점이 아쉽습니다. 원문에서 언급된 '기술 경쟁력 확보, 고객사 확대, 수익구조 개선 등 실질적 사업성과로 뒷받침돼야 한다는 지적' 등의 중요한 내용을 요약에 포함시키면 더욱 완벽한 요약이 될 것 같습니다. 이 부분을 고려하여 요약을 수정해보시는 것을 추천드립니다.
✅ '정확성' 평가 통과
❌ '포괄성' 평가에서 좋음이 아님
[should_stop] next_step = no


KeyError: 'Input to PromptTemplate is missing variables {\'"foo"\', \'"properties"\'}.  Expected: [\'"foo"\', \'"properties"\', \'article\', \'feedback\', \'summary\'] Received: [\'article\', \'summary\', \'feedback\']\nNote: if you intended {"foo"} to be part of the string and not a variable, please escape it with double curly braces like: \'{{"foo"}}\'.\nFor troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/INVALID_PROMPT_INPUT '

In [35]:
def summarize_top_articles(total_contents :str):

    agent = NewsSummaryAgent()
    runnable = build_summary_graph(agent)
    results = []
    
    doc = total_contents

    state = {
        "article": doc['content'],
        "summary": "",
        "feedback": "",
        "iteration": 0
    }
    result = runnable.invoke(state)


    print("✅ 실행 결과 타입:", type(result))
    print("✅ 실행 결과 키 목록:", result.keys())
    print("✅ 실행 결과 전체 내용:", result)

    if "final_summary" not in result:
        print("❌ final_summary 키가 없습니다.")

    print(f'실행 결과 : {result}')
    results.append({
        "ticker": doc['ticker'],
        "date": doc['Date'],
        "header": doc['title'],
        "url": doc['url'],
        "summary": result["final_summary"], 
            "feedback": result.get("last_feedback", "피드백 없음")
    })
    return pd.DataFrame(results)
